# Caroline: production planning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/caroline-production-planning.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/caroline-production-planning.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## The production-planning problem

Caroline owns a company that produces trophies for football and golf events. Each product consumes raw materials and yields a specific profit per unit:

- football trophy: wood base, engraved plaque, and brass football; €12 profit and 4 dm of wood;
- golf trophy: wood base, engraved plaque, and golf ball; €9 profit and 2 dm of wood.

Caroline's current stock consists of 1,000 footballs, 1,500 golf balls, 1,750 plaques, and 480 m (4,800 dm) of wood.

> How many football trophies and golf trophies should Caroline produce to maximize her profit while respecting the available raw materials?

Caroline has two decisions:

- $x_1$: the number of football trophies to produce;
- $x_2$: the number of golf trophies to produce.

The resulting optimization model is

$$
\begin{array}{rrcrcl}
\max    & 12x_1 & + & 9x_2               \\
\text{s.t.} &   x_1 &   &      & \leq & 1000 \\
        &       &   &  x_2 & \leq & 1500 \\
        &   x_1 & + &  x_2 & \leq & 1750 \\
        &  4x_1 & + & 2x_2 & \leq & 4800 \\
        &   x_1 & , &  x_2 & \geq & 0.   \\
\end{array}
$$


## Setup
This lesson compares Ipopt, CBC and HiGHS on the same model. Install missing Python packages and the open-source COIN solver binaries used by Ipopt and CBC. Existing packages are kept. Solver installation is separate from importing Pyomo.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pyomo': 'pyomo', 'highspy': 'highspy', 'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
ensure_packages(required_packages)
from teaching_utils import install_coin_solvers, make_solver, solve_checked
install_coin_solvers()
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = solver_name = 'appsi_highs'


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


We can inspect registered solver interfaces with Pyomo's command-line help, or query named interfaces through Python. Near the end, we implement that Python check ourselves. Availability means an interface can be constructed; an actual solve also checks that its executable and licence work.


In [ ]:
from pyomo.scripting.pyomo_main import main
main(['help', '--solvers'])


## Dependencies

We are now ready to focus on Caroline's production planning story.

Below you see the dependencies of this notebook.

In [ ]:
import pandas as pd
import numpy as np
import fractions
import pyomo.environ as pyo
import matplotlib.pyplot as plt


## Recall the model

$$
\begin{array}{rrcrcl}
\max    & 12x_1 & + & 9x_2               \\
s.t.    &   x_1 &   &      & \leq & 1000 \\
        &       &   &  x_2 & \leq & 1500 \\
        &   x_1 & + &  x_2 & \leq & 1750 \\
        &  4x_1 & + & 2x_2 & \leq & 4800 \\
        &   x_1 & , &  x_2 & \geq & 0    \\
\end{array}
$$

## Very first version

In [ ]:
def CreateFirstVersionOfCaroline():
    model = pyo.ConcreteModel(name='Caroline')

    # Declare variables
    model.x1 = pyo.Var()
    model.x2 = pyo.Var()

    # Objective: Maximize profit
    @model.Objective(sense=pyo.maximize)
    def profit(model):
        return 12 * model.x1 + 9 * model.x2

    # Constraints
    @model.Constraint()
    def footballs(model):
        return model.x1 <= 1000

    @model.Constraint()
    def golf_balls(model):
        return model.x2 <= 1500

    @model.Constraint()
    def plaques(model):
        return model.x1 + model.x2 <= 1750

    @model.Constraint()
    def wood(model):
        return 4 * model.x1 + 2 * model.x2 <= 4800

    @model.Constraint()
    def x1domain(model):
        return model.x1 >= 0

    @model.Constraint()
    def x2domain(model):
        return model.x2 >= 0

    return model


In [ ]:
first = CreateFirstVersionOfCaroline()
first.pprint()


Ipopt solves smooth continuous nonlinear models. We first use it on this linear model to compare its numerical result with solvers specialized for linear optimization. Each run creates a fresh model and solver so previous values do not affect the comparison.


In [ ]:
from time import perf_counter
comparison_rows = []
first = CreateFirstVersionOfCaroline()
started = perf_counter()
results = solve_checked(first, 'ipopt')
elapsed = perf_counter() - started
comparison_rows.append({'solver': 'ipopt', 'objective': pyo.value(first.profit),
                        'football': pyo.value(first.x1), 'golf': pyo.value(first.x2),
                        'seconds': elapsed, 'termination': str(results.solver.termination_condition)})
print('ipopt', results.solver.status, results.solver.termination_condition)
first.display()
assert abs(pyo.value(first.profit) - 17700) < 0.02
assert abs(pyo.value(first.x1) - 650) < 0.02
assert abs(pyo.value(first.x2) - 1100) < 0.02


CBC and HiGHS can exploit the linear structure. Compare the objective, variable values and elapsed times with Ipopt. All three should reach the same optimum within numerical tolerance. This tiny example demonstrates interfaces; its timings are not a reliable performance ranking.


In [ ]:
first = CreateFirstVersionOfCaroline()
started = perf_counter()
results = solve_checked(first, 'cbc')
elapsed = perf_counter() - started
comparison_rows.append({'solver': 'cbc', 'objective': pyo.value(first.profit),
                        'football': pyo.value(first.x1), 'golf': pyo.value(first.x2),
                        'seconds': elapsed, 'termination': str(results.solver.termination_condition)})
print('cbc', results.solver.status, results.solver.termination_condition)
first.display()
assert abs(pyo.value(first.profit) - 17700) < 0.02
assert abs(pyo.value(first.x1) - 650) < 0.02
assert abs(pyo.value(first.x2) - 1100) < 0.02


In [ ]:
first = CreateFirstVersionOfCaroline()
started = perf_counter()
results = solve_checked(first, 'appsi_highs')
elapsed = perf_counter() - started
comparison_rows.append({'solver': 'appsi_highs', 'objective': pyo.value(first.profit),
                        'football': pyo.value(first.x1), 'golf': pyo.value(first.x2),
                        'seconds': elapsed, 'termination': str(results.solver.termination_condition)})
print('appsi_highs', results.solver.status, results.solver.termination_condition)
first.display()
assert abs(pyo.value(first.profit) - 17700) < 0.02
assert abs(pyo.value(first.x1) - 650) < 0.02
assert abs(pyo.value(first.x2) - 1100) < 0.02

comparison = pd.DataFrame(comparison_rows)
assert comparison['solver'].nunique() == 3
display(comparison)


### Examining slacks

Constraints have `.lslack()` and `.uslack()`, and since our constraints are of type $\leq$ we are interested in the upper slack, i.e. `.uslack()`.

In [ ]:
print( f'{first.footballs.uslack()=}\n{first.golf_balls.uslack()=}\n{first.plaques.uslack()=}\n{first.wood.uslack()=}' )


## Nature of the variables: a second version

Although the first model is correct, we may notice that the variables are bounded.

This saves $4$ simple constraints: $2$ per variable.

In [ ]:
def CreateSecondVersionOfCaroline():
    model = pyo.ConcreteModel(name='Caroline')

    # Declare variables with bounds
    model.x1 = pyo.Var(bounds=(0, 1000))
    model.x2 = pyo.Var(bounds=(0, 1500))

    # Objective: Maximize profit
    @model.Objective(sense=pyo.maximize)
    def profit(model):
        return 12 * model.x1 + 9 * model.x2

    # Constraints
    @model.Constraint()
    def plaques(model):
        return model.x1 + model.x2 <= 1750

    @model.Constraint()
    def wood(model):
        return 4 * model.x1 + 2 * model.x2 <= 4800

    return model


In [ ]:
second = CreateSecondVersionOfCaroline()
second.pprint()


In [ ]:
%time results = pyo.SolverFactory(SOLVER).solve(second)
print(results.solver.status, results.solver.termination_condition )
print(second.profit.expr())

second.display()


### Examining slacks

Constraints have `.lslack()` and `.uslack()` but variables do not.

In [ ]:
print( f'{second.x1.ub-second.x1()=}\n{second.x2.ub-second.x2()=}\n{second.plaques.uslack()=}\n{second.wood.uslack()=}' )


## A version that makes better use of data

Remember where you are: the `python` ecosystem. As you know, this is a great place for data science, so despite your passion being on mathematical optimization, please open your eyes for the great toolset that data scientists love and use.

Let us start by organizing the data of our problem in a simple `data frame`.

In [ ]:
def GetNominalData():
    data = pd.DataFrame()
    data.at['Football','profit']    = 12
    data.at['Football','wood']      =  4
    data.at['Football','plaque']    =  1
    data.at['Football','football']  =  1
    data.at['Golf','profit']        =  9
    data.at['Golf','wood']          =  2
    data.at['Golf','plaque']        =  1
    data.at['Golf','golf ball']     =  1

    available = pd.DataFrame()
    available.at['available','wood']     =  4800
    available.at['available','plaque']   =  1750
    available.at['available','Football'] =  1000
    available.at['available','Golf']     =  1500

    return data.fillna(0).astype(int), available.astype(int)

trophies, materials = GetNominalData()


In [ ]:
trophies


In [ ]:
materials


The index of this data frame is suited as index for our variables as well.
The columns `profit` and `wood` are directly usable to express the objective function and the wood consumption constraint.

In [ ]:
def CreateThirdVersionOfCaroline(trophies, materials):
    model = pyo.ConcreteModel(name='Caroline')

    # Declare sets and variables
    model.trophies = pyo.Set(initialize=trophies.index)
    model.x = pyo.Var(model.trophies, within=pyo.NonNegativeReals)

    # Objective: Maximize profit
    @model.Objective(sense=pyo.maximize)
    def profit(model):
        return pyo.quicksum(trophies['profit'][t] * model.x[t] for t in model.trophies)

    # Constraints
    @model.Constraint()
    def footballs(model):
        return model.x['Football'] <= materials['Football']['available']

    @model.Constraint()
    def golf_balls(model):
        return model.x['Golf'] <= materials['Golf']['available']

    @model.Constraint()
    def wood(model):
        return pyo.quicksum(trophies['wood'][t] * model.x[t] for t in model.trophies) <= materials['wood']['available']

    @model.Constraint()
    def plaques(model):
        return pyo.quicksum(model.x[t] for t in model.trophies) <= materials['plaque']['available']

    return model


In [ ]:
third = CreateThirdVersionOfCaroline(trophies,materials)
third.pprint()


In [ ]:
%time results = pyo.SolverFactory(SOLVER).solve(third)
print(results.solver.status, results.solver.termination_condition )

print(third.profit())

third.display()


The solutions are also (computed) data, and hence can be maintained and processed in data frames.

In [ ]:
def AddSolution( solutions, trophies, model, name ):
    for t in trophies.index:
        solutions.at[t,name] = model.x[t].value
    solutions.at['value',name] = model.profit.expr()
    return solutions


In [ ]:
solutions = pd.DataFrame()
AddSolution( solutions, trophies, third, 'optimal' )


In [ ]:
materials


In [ ]:
materials.at['available','plaque'] += 1
materials


In [ ]:
modified = CreateThirdVersionOfCaroline(trophies,materials)
pyo.SolverFactory(SOLVER).solve(modified)
AddSolution( solutions, trophies, modified, 'added 1 plaque' )
solutions


In [ ]:
materials.at['available','wood'] += 5
modified = CreateThirdVersionOfCaroline(trophies,materials)
pyo.SolverFactory(SOLVER).solve(modified)
AddSolution( solutions, trophies, modified, 'added 5 to wood' )
solutions


Notice how some modiffications yield noninteger solutions!

### Examining slacks

In [ ]:
print( f'{third.footballs.uslack()=}\n{third.golf_balls.uslack()=}\n{third.plaques.uslack()=}\n{third.wood.uslack()=}' )


In [ ]:
third.footballs.display()


In [ ]:
third.golf_balls.display()


In [ ]:
third.plaques.display()


In [ ]:
third.wood.display()


# Optional Materials

The following materials extend beyond the basic use of Pyomo, covering advanced topics in both programming and optimization. Specifically, you have not yet been introduced to the dual problem and dual solutions. As such, this section is considered supplementary and optional.

Note that some of the functions added here are uefull in general and independent of Caroline's production planning.

## Examining `pyomo` models

`pyomo` allows maintains models in vary generic ways and allows them to be accessed and discovered even if we did not code their creation ourselves.

In [ ]:
def ShowModelComponents( model ):
    for v in model.component_objects(pyo.Var, active=True):
        print ("Variable  ",v)
        varobject = getattr(model, str(v))
        for index in varobject:
            print ("     ",index, varobject[index].value)
    for o in model.component_objects(pyo.Objective, active=True):
        print ("Objective ",o)
        varobject = getattr(model, str(o))
        for index in varobject:
            print ("    ",index, varobject[index].expr())
    for c in model.component_objects(pyo.Constraint, active=True):
        print ("Constraint",c)
        varobject = getattr(model, str(c))
        for index in varobject:
            print ("     ",index, varobject[index].uslack())


In [ ]:
ShowModelComponents( first )


In [ ]:
ShowModelComponents( second )


In [ ]:
ShowModelComponents( third )


## Resetting models
Here we explain the reset code, so its implementation is visible. Clearing unfixed variable values also works for variables inside nested blocks. It does not clear a persistent solver's internal state; use a fresh model and solver for independent comparisons. Other notebooks can import `reset_model` from the shared teaching utilities.


In [ ]:
def ResetModel(model):
    # Preserve fixed decisions, and include variables inside nested blocks.
    for variable in model.component_data_objects(pyo.Var, descend_into=True):
        if not variable.fixed:
            variable.set_value(None)


In [ ]:
model = CreateFirstVersionOfCaroline()


In [ ]:
first.pprint()


In [ ]:
pyo.SolverFactory(SOLVER).solve(first)


In [ ]:
first.pprint()


In [ ]:
ResetModel(first)


In [ ]:
first.pprint()


## List of available solvers
Check a named selection of Pyomo interfaces directly. We keep the code here because this section explains it; other notebooks import `available_pyomo_solvers` from the shared utilities. This is a course selection, not every registered plugin. Different interfaces to HiGHS are still one solver engine.


In [ ]:
def ListAvailableSolvers(candidates=('appsi_highs', 'ipopt', 'cbc', 'glpk', 'gurobi_direct')):
    availability = {}
    for name in candidates:
        availability[name] = bool(make_solver(name).available(exception_flag=False))
    return availability


In [ ]:
ListAvailableSolvers()


## Requesting and obtaining dual solutions

In [ ]:
def ShowDuals( model ):
    # display all duals
    print ("Duals")
    for c in model.component_objects(pyo.Constraint, active=True):
        print ("Constraint ",c)
        for index in c:
            print ("      ", index, str(fractions.Fraction(model.dual[c[index]])))


In [ ]:
third.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

%time results = pyo.SolverFactory(SOLVER).solve(third)
print(results.solver.status, results.solver.termination_condition )


In [ ]:
ShowDuals( third )


## A final version of Caroline's model with indexed variables and variable upper bounds

Using a rule for bounds allows you to define variable upper bounds for indexed variables.

In [ ]:
def CreateFourthVersionOfCaroline(trophies, materials):
    model = pyo.ConcreteModel(name='Caroline')

    # Declare sets and variables with bounds
    model.trophies = pyo.Set(initialize=trophies.index)

    @model.Var(model.trophies, bounds=lambda model, trophy: (0, materials[trophy]['available']))
    def x(model, trophy):
        return 0

    # Objective: Maximize profit
    @model.Objective(sense=pyo.maximize)
    def profit(model):
        return pyo.quicksum(trophies['profit'][t] * model.x[t] for t in model.trophies)

    # Constraints using decorators
    @model.Constraint()
    def plaques(model):
        return pyo.quicksum(model.x[t] for t in model.trophies) <= materials['plaque']['available']

    @model.Constraint()
    def wood(model):
        return pyo.quicksum(trophies['wood'][t] * model.x[t] for t in model.trophies) <= materials['wood']['available']

    return model


In [ ]:
fourth = CreateFourthVersionOfCaroline(trophies,materials)
fourth.pprint()


In [ ]:
%time results = pyo.SolverFactory(SOLVER).solve(fourth)
print(results.solver.status, results.solver.termination_condition )

print(fourth.profit.expr())

fourth.display()


In [ ]:
fourth.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

%time results = pyo.SolverFactory(SOLVER).solve(fourth)
print(results.solver.status, results.solver.termination_condition )


In [ ]:
ShowDuals( fourth )


## Check the original model
The original resource limits must give the same optimum in its scalar and indexed formulations. Altering the data above changes the later scenarios, not this baseline.


In [ ]:
baseline = CreateFirstVersionOfCaroline()
result = pyo.SolverFactory(SOLVER).solve(baseline)
assert_optimal_termination(result)
assert abs(pyo.value(baseline.profit) - 17700) < 1e-6
assert abs(pyo.value(baseline.x1) - 650) < 1e-6
assert abs(pyo.value(baseline.x2) - 1100) < 1e-6


## Explain it without the notebook

Close the code and explain one example in your own words. Predict the effect of a small change before testing it.

> **Optional UvA AI Chat prompt:** Act as a Socratic tutor. Ask me one question at a time about the example I paste below. First ask for my prediction and reasoning. Give a small hint if I am stuck; do not produce a complete solution unless I explicitly ask after attempting it. Check my explanation, not just my final number.

An AI response is not evidence that a result is correct. Verify it with the mathematical model and independent checks. See the [Socratic method](https://en.wikipedia.org/wiki/Socratic_method).
